# STATE SE — PBMC 2154 cells, quality 0.4751547

Prepare data, train, embed, and compute LMI (protein_counts only).

In [8]:
import sys, os
sys.path.insert(0, os.path.expanduser(
    "~/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/scaling_laws/src"
))

from pathlib import Path
import glob
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import umap

from scaling_laws.prepare.data import Experiments, PrepareData
from scaling_laws.algo import State

In [9]:
DATA_DIR = Path(os.path.expanduser("~/noise_scaling/data"))
DATASET = "PBMC"
SIZE = 2154
QUALITY = 0.4751547
DEVICE = 4
SEED = 42

In [10]:
# --- Gene count check: raw dataset vs ESM embeddings ---
import torch

raw_path = DATA_DIR / DATASET / "raw" / "raw.h5ad"
adata_raw = ad.read_h5ad(raw_path, backed="r")
raw_genes = set(adata_raw.var_names)
n_raw = len(raw_genes)
adata_raw.file.close()

esm_emb = torch.load(DATA_DIR / "other" / "esm" / "merged_esm_embeddings.pt", map_location="cpu")
esm_genes = set(esm_emb.keys())
n_esm = len(esm_genes)
del esm_emb

overlap = len(raw_genes & esm_genes)
print(f"{DATASET}: {overlap}/{n_raw} raw genes found in ESM ({100*overlap/n_raw:.1f}%), ESM total: {n_esm}")

[2026-04-14 16:34:23] PBMC: 14899/20729 raw genes found in ESM (71.9%), ESM total: 61260


## 1. Prepare STATE data

Run `state emb preprocess` to build the gene-embedding profile.

In [11]:
experiments = Experiments(
    path_to_data_dir=str(DATA_DIR),
    datasets=[DATASET],
    qualities=[QUALITY],
    sizes=[SIZE],
    algos=["State"],
    signal_columns=["protein_counts"],
    device=DEVICE,
)

experiments.prepare_state_data()

[2026-04-14 16:34:23]   Using ESM embeddings: /home/igor/noise_scaling/data/other/esm/merged_esm_embeddings.pt
[2026-04-14 16:34:23] 
=== Preparing State data for PBMC / 2154 / 0.4751547 ===
[2026-04-14 16:34:23]   State train manifest: /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/train.csv
[2026-04-14 16:34:23]   State val manifest: /home/igor/noise_scaling/data/PBMC/validation/0.4751547/preprocessed/state_data/val.csv
[2026-04-14 16:34:23]   State test manifest: /home/igor/noise_scaling/data/PBMC/test/0.4751547/preprocessed/state_data/test.csv
[2026-04-14 16:34:23]   Running: /home/igor/miniconda3/envs/state/bin/python -m state emb preprocess --profile-name scaling_PBMC_2154_0_4751547 --train-csv /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/train.csv --val-csv /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/val_combined.csv --output-dir /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/st

2026-04-14 16:34:24,897 INFO: Loading existing embeddings from /home/igor/noise_scaling/data/other/esm/merged_esm_embeddings.pt
2026-04-14 16:34:27,463 INFO: Loading training and validation CSV files...
2026-04-14 16:34:27,467 INFO: Processing 1 training datasets and 2 validation datasets...
2026-04-14 16:34:27,471 INFO: Scanning datasets serially...
Scanning datasets: 100%|██████████| 3/3 [00:09<00:00,  3.09s/it]
2026-04-14 16:34:36,753 INFO: Found 20729 unique genes across datasets
2026-04-14 16:34:38,097 INFO: Saved embeddings to /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/all_embeddings_scaling_PBMC_2154_0_4751547.pt
2026-04-14 16:34:38,123 INFO: Saved dataset mapping to /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/ds_emb_mapping_scaling_PBMC_2154_0_4751547.torch
2026-04-14 16:34:38,124 INFO: Saved valid gene masks to /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/valid_genes_masks_scaling_PBMC_21

[2026-04-14 16:34:38]   Config patched: val uses val-only (no test leakage)
[2026-04-14 16:34:38]   Profile saved to /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data


In [12]:
# Verify: list the generated state_data/ contents
state_data = DATA_DIR / DATASET / str(SIZE) / str(QUALITY) / "preprocessed" / "state_data"
print(f"Profile dir: {state_data}")
for f in sorted(state_data.iterdir()):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name:60s}  {size_mb:.2f} MB" if f.is_file() else f"  {f.name}/")

# Also check val / test state_data dirs were created
for split in ["validation", "test"]:
    sd = DATA_DIR / DATASET / split / str(QUALITY) / "preprocessed" / "state_data"
    print(f"\n{split} state_data: {sd}  (exists={sd.exists()})")
    if sd.exists():
        for f in sorted(sd.iterdir()):
            print(f"  {f.name}")

[2026-04-14 16:34:38] Profile dir: /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data
[2026-04-14 16:34:38]   all_embeddings_scaling_PBMC_2154_0_4751547.pt                 249.23 MB
[2026-04-14 16:34:38]   ds_emb_mapping_scaling_PBMC_2154_0_4751547.torch              0.50 MB
[2026-04-14 16:34:38]   state_config.yaml                                             0.01 MB
[2026-04-14 16:34:38]   train.csv                                                     0.00 MB
[2026-04-14 16:34:38]   train_scaling_PBMC_2154_0_4751547.csv                         0.00 MB
[2026-04-14 16:34:38]   val_combined.csv                                              0.00 MB
[2026-04-14 16:34:38]   val_only.csv                                                  0.00 MB
[2026-04-14 16:34:38]   val_only_scaling_PBMC_2154_0_4751547.csv                      0.00 MB
[2026-04-14 16:34:38]   val_scaling_PBMC_2154_0_4751547.csv                           0.00 MB
[2026-04-14 16:34:38]   valid_genes_masks_s

## 2. Train

In [13]:
base_dir = DATA_DIR / DATASET / str(SIZE) / str(QUALITY)

model = State(
    base_dir=str(base_dir),
    device=DEVICE,
    max_epochs=max(1, int(10 * (100_000 / SIZE))),
    early_stopping_patience=3,
    dataset_name=DATASET,
    seed=SEED,
    pad_length=2048,
)

print(f"Train:      {model.train_data_path / 'preprocessed.h5ad'}")
print(f"Validation: {model.validation_data_path / 'preprocessed.h5ad'}")
print(f"Test:       {model.test_data_path / 'preprocessed.h5ad'}")
print(f"Profile:    {model.profile_dir}")
print(f"Config:     {model.config_path}")
print(f"pad_length: {model.pad_length}, P/N/S: {model.pad_length // 4}")

[2026-04-14 16:34:39] Using GPU 4 (visible as cuda:0)
[2026-04-14 16:34:39] Train:      /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/preprocessed.h5ad
[2026-04-14 16:34:39] Validation: /home/igor/noise_scaling/data/PBMC/validation/0.4751547/preprocessed/preprocessed.h5ad
[2026-04-14 16:34:39] Test:       /home/igor/noise_scaling/data/PBMC/test/0.4751547/preprocessed/preprocessed.h5ad
[2026-04-14 16:34:39] Profile:    /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data
[2026-04-14 16:34:39] Config:     /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/state_config.yaml
[2026-04-14 16:34:39] pad_length: 2048, P/N/S: 512


In [14]:
model.train()

[2026-04-14 16:34:39]   Data: 2154 cells, ~33 batches/epoch, val every 33 steps
[2026-04-14 16:34:39]   Running: /home/igor/miniconda3/envs/state/bin/python -m state emb fit --conf /home/igor/noise_scaling/data/PBMC/2154/0.4751547/preprocessed/state_data/state_config.yaml embeddings.current=scaling_PBMC_2154_0_4751547 dataset.current=scaling_PBMC_2154_0_4751547 dataset.num_cells=2154 dataset.num_train_workers=4 dataset.num_val_workers=2 dataset.pad_length=2048 dataset.P=512 dataset.N=512 dataset.S=512 model.batch_size=64 model.emsize=256 model.d_hid=512 model.nhead=4 model.nlayers=3 model.output_dim=256 model.dataset_correction=false model.dropout=0.1 optimizer.max_lr=0.0001 optimizer.gradient_accumulation_steps=1 optimizer.weight_decay=0.01 experiment.name=state_scaling_PBMC_2154_0_4751547 experiment.num_epochs=464 experiment.num_gpus_per_node=1 experiment.num_nodes=1 experiment.port=52559 experiment.val_check_interval=33 experiment.limit_val_batches=50 experiment.checkpoint.path=/hom

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4]
Loading `train_dataloader` to estimate number of stepping batches.


INFO:state.emb.train.callbacks:CumulativeFLOPSCallback: Measured FLOPs per  0.00it/s v_num: 0.000it/s 
batch: 6402360022656
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:05 • 0:00:01 7.44it/s v_num: 0.0000.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:05 • 0:00:01 7.44it/s  v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:05 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:05 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.44it/s v_num: 0.000
Epoch 0/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:06 • 0:00:01 7.

Metric validation/val_loss improved. New best score: 20.572


Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:55 • -:--:-- 0.00it/s v_num: 0.0000.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s  v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:56 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:57 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:57 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:57 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:00:57 • -:--:-- 0.00it/s v_num: 0.000
Epoch 1/463 ━━━━━━━━━━━

...

Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:55 • 0:00:01 7.49it/s v_num: 0.0000.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:55 • 0:00:01 7.49it/s  v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:55 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:55 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:55 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:55 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:56 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:56 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:56 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:56 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:56 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━━━━━━━━━━━╺ 33/34 0:01:56 • 0:00:01 7.49it/s v_num: 0.000
Epoch 2/463 ━━━━━━━━━━━

Metric validation/val_loss improved by 0.598 >= min_delta = 0.0. New best score: 19.057


Epoch 3/463 ━━━━━━━━━━━━━━━━━━━╺━━ 30/34 0:01:48 • 0:00:01 7.32it/s v_num: 0.000



Detected KeyboardInterrupt, attempting graceful shutdown ...


KeyboardInterrupt: 

## 3. Training / validation loss curves

In [ ]:
log_dirs = sorted(glob.glob(
    str(model.checkpoint_dir / f"state_{model.profile_name}" / "version_*")
))
metrics_file = Path(log_dirs[-1]) / "metrics.csv"
print(f"Reading: {metrics_file}")

df = pd.read_csv(metrics_file)
train_loss = df[["step", "trainer/train_loss"]].dropna()
val_loss = df[["step", "validation/val_loss"]].dropna()

fig, ax = plt.subplots(figsize=(8, 4))

# Plot train loss as connected blue line
ax.plot(
    train_loss["step"], train_loss["trainer/train_loss"],
    label="train loss", color="blue", linewidth=1
)

# Plot val loss as connected orange line with marker 'o'
ax.plot(
    val_loss["step"], val_loss["validation/val_loss"],
    color="orange", marker="o", label="val loss", markersize=4, linewidth=1
)

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title(f"STATE SE — PBMC {SIZE} cells, q={QUALITY} (log-log)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3, which="both", linestyle='--')
fig.tight_layout()
plt.show()

## 4. Embed test set

In [ ]:
embeddings = model.embed()
print(f"Embeddings shape: {embeddings.shape}")

## 5. Compute LMI mutual information (protein_counts only)

In [ ]:
model.signal_columns = ["protein_counts"]
mi_results = model.mutual_information(max_epochs=300)
print("\nLMI results:")
for signal, mi in mi_results.items():
    print(f"  {signal}: {mi:.5f}")

## 6. Compare LMI across methods

In [ ]:
results_root = base_dir / "results"
algos = ["PCA", "RandomProjection", "SCVI", "Geneformer", "State"]

rows = []
for algo in algos:
    sig = "Y_protein_counts_1.0_geneformer" if algo == "Geneformer" else "Y_protein_counts_1.0"
    mi_base = results_root / algo / "model" / "MI"
    if not mi_base.exists():
        print(f"{algo:20s}  NOT FOUND")
        continue
    for seed_dir in sorted(mi_base.iterdir()):
        mi_file = seed_dir / sig / "lmi_mutual_information.txt"
        if mi_file.exists():
            mi = float(mi_file.read_text().strip())
            rows.append({"Algorithm": algo, "seed": int(seed_dir.name), "LMI (protein_counts)": mi})
            print(f"{algo:20s}  seed={seed_dir.name}  {mi:.5f}")

scores = pd.DataFrame(rows)
scores_agg = (
    scores.groupby("Algorithm")["LMI (protein_counts)"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_lmi", "std": "std_lmi", "count": "n_seeds"})
    .reset_index()
    .sort_values("mean_lmi", ascending=False)
)
scores_agg["std_lmi"] = scores_agg["std_lmi"].fillna(0)
scores_agg

In [ ]:
colors = {"PCA": "#4C72B0", "RandomProjection": "#DD8452",
          "SCVI": "#55A868", "Geneformer": "#C44E52", "State": "#8172B3"}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    scores_agg["Algorithm"], scores_agg["mean_lmi"],
    yerr=scores_agg["std_lmi"],
    capsize=4,
    color=[colors.get(a, "#999") for a in scores_agg["Algorithm"]],
    edgecolor="black", linewidth=0.5,
)

for bar, mean, std, n in zip(bars, scores_agg["mean_lmi"], scores_agg["std_lmi"], scores_agg["n_seeds"]):
    label = f"{mean:.3f}"
    if n > 1:
        label += f"\n\u00b1{std:.3f} (n={n})"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + std + 0.02,
            label, ha="center", va="bottom", fontsize=9)

ax.set_ylabel("LMI Mutual Information (protein_counts)")
ax.set_title(f"PBMC {SIZE} cells, quality {QUALITY} — LMI comparison")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, (scores_agg["mean_lmi"] + scores_agg["std_lmi"]).max() * 1.2)
fig.tight_layout()
plt.show()

## 7. UMAP of State embeddings

In [ ]:
test_h5ad = model.test_data_path / "preprocessed.h5ad"
adata_test = ad.read_h5ad(test_h5ad)

max_cells = 10_000
if embeddings.shape[0] > max_cells:
    rng = np.random.default_rng(42)
    idx = rng.choice(embeddings.shape[0], max_cells, replace=False)
    emb_sub = embeddings[idx]
else:
    emb_sub = embeddings

print(f"Running UMAP on {emb_sub.shape[0]} cells, {emb_sub.shape[1]} dims")
reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
umap_coords = reducer.fit_transform(emb_sub)

# Just plot the embeddings with no coloring
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(umap_coords[:, 0], umap_coords[:, 1], color="#888", s=2, alpha=0.7)
ax.set_title("UMAP of STATE SE embeddings")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

fig.suptitle(f"STATE SE embeddings — PBMC {SIZE} cells, q={QUALITY}", fontsize=13)
fig.tight_layout()
plt.show()

- check if mode collapse show on un trained model -> it should be close to random projection!
- space out the validation loss computation
- decrease the model dimensions (?)
- 